# Parte 3 — Limpieza y calidad de datos en Google Colab

1. En Colab, selecciona **Entorno de ejecución → Ejecutar todas**.
2. Cuando aparezca el selector, sube únicamente `mercadolibre_raw.csv`.
3. Al terminar se descargará un ZIP con el CSV limpio, la sección del informe y las tablas de control.

El proceso usa pandas, incluido en Colab. No vuelve a ejecutar el scraper ni consulta Mercado Libre. Conserva los valores ausentes; no inventa precios anteriores ni vendedores.

In [ ]:
from google.colab import files
from pathlib import Path

print('Selecciona mercadolibre_raw.csv desde tu computadora.')
archivos = files.upload()
csv_subidos = [nombre for nombre in archivos if nombre.lower().endswith('.csv')]
if len(csv_subidos) != 1:
    raise ValueError('Sube solamente el archivo mercadolibre_raw.csv y vuelve a ejecutar esta celda.')
Path('/content/mercadolibre_raw.csv').write_bytes(archivos[csv_subidos[0]])
print('CSV listo para procesar.')

In [ ]:
from pathlib import Path
from urllib.parse import urlsplit, parse_qs
import re
import pandas as pd

RUTA = Path('/content/mercadolibre_raw.csv')
SALIDA = Path('/content/resultados_parte3')
SALIDA.mkdir(exist_ok=True)
raw = pd.read_csv(RUTA, encoding='utf-8-sig', dtype='string')
df = raw.copy()
df.insert(0, 'fila_csv', range(2, len(df) + 2))
print('Dimensiones originales:', raw.shape)
print(raw.isna().mean().mul(100).round(2).to_string())

## Normalización e identificadores
Los precios usan punto para miles y coma para decimales: `S/ 4.849` se convierte en `4849.00` y `S/ 987,90` en `987.90`.

El identificador original puede proceder de `/p/MPE...`, mientras la misma URL contiene otro código en `wid`. Para identificar la oferta se prioriza `wid` y, si falta, el código de una ruta directa que no contenga `/p/`. Se mantiene el identificador del scraper como `id_extraido`. Las filas sin un identificador de oferta recuperable se separan para revisión; no se equipara automáticamente un código de catálogo a una publicación. Esta regla se basa en la estructura del CSV y no implica una verificación en vivo de las ofertas.

In [ ]:
for c in raw.columns:
    df[c] = df[c].str.replace(r'\s+', ' ', regex=True).str.strip().replace('', pd.NA)

def oferta_id(url):
    if pd.isna(url):
        return pd.NA
    u = urlsplit(url)
    parametros = parse_qs(u.query + '&' + u.fragment)
    wid = parametros.get('wid', [''])[0]
    if re.fullmatch(r'MPE[0-9]+', wid):
        return wid
    m = re.search(r'/MPE-?([0-9]+)(?:-|/|$)', u.path)
    if m and '/p/' not in u.path:
        return 'MPE' + m.group(1)
    return pd.NA

def precio(serie):
    s = serie.str.replace(r'^S/\s*', '', regex=True).str.replace(' ', '', regex=False)
    valido = s.str.fullmatch(r'(?:[0-9]{1,3}(?:\.[0-9]{3})+|[0-9]+)(?:,[0-9]{1,2})?', na=False)
    return pd.to_numeric(s.where(valido).str.replace('.', '', regex=False).str.replace(',', '.', regex=False), errors='coerce')

df = df.rename(columns={'id_publicacion': 'id_extraido'})
df['id_publicacion'] = df['url_publicacion'].map(oferta_id).astype('string')
df['precio_actual'] = precio(df['precio_actual_texto'])
df['moneda'] = df['moneda'].replace({'S/': 'PEN'})
df['numero_cuotas'] = pd.to_numeric(df['cuotas_texto'].str.extract(r'^(\d+) cuotas? de ', expand=False), errors='coerce').astype('Int64')
importe = df['cuotas_texto'].str.extract(r'cuotas? de (S/ [\d., ]+)', expand=False).str.strip()
df['monto_cuota'] = precio(importe)
for c in ['pagina', 'posicion_pagina']:
    df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int64')
df['fecha_extraccion'] = pd.to_datetime(df['fecha_extraccion'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
print('IDs originales diferentes del ID de oferta:', int((df.id_extraido != df.id_publicacion).fillna(False).sum()))
print('Filas sin ID de oferta recuperable:', int(df.id_publicacion.isna().sum()))
print('Precios actuales no interpretables:', int(df.precio_actual.isna().sum()))

## Calidad y duplicados
Se separan registros sin campos esenciales válidos. Para ofertas repetidas se conserva la observación más reciente y, en empates, la última fila del archivo. No se combinan atributos de observaciones distintas. El listado es una muestra de resultados de búsquedas; `categoria` representa la búsqueda de origen, que puede incluir accesorios u otros productos.

Las cuotas ausentes permanecen nulas. Un texto ausente no demuestra ausencia de financiamiento. Tampoco se asume que un precio anterior ausente sea igual al precio actual.

In [ ]:
invalidas = (df['id_publicacion'].isna() | df['titulo'].isna() | df['categoria'].isna()
             | df['precio_actual'].isna() | df['precio_actual'].le(0)
             | ~df['moneda'].eq('PEN') | df['fecha_extraccion'].isna()
             | df['pagina'].isna() | df['posicion_pagina'].isna())
revision = df.loc[invalidas].copy()
revision.to_csv(SALIDA / 'filas_para_revision.csv', index=False, encoding='utf-8')
validas = df.loc[~invalidas].sort_values(['fecha_extraccion', 'fila_csv'], kind='stable')
repetidas = validas.duplicated('id_publicacion', keep='last')
validas.loc[repetidas].to_csv(SALIDA / 'duplicados_retirados.csv', index=False, encoding='utf-8')
unicas = validas.loc[~repetidas].sort_values('fila_csv').copy()

# Conservamos atributos con cobertura suficiente. Los textos crudos quedan en el CSV original.
columnas = ['id_publicacion', 'id_extraido', 'categoria', 'titulo', 'moneda',
            'precio_actual', 'numero_cuotas', 'monto_cuota', 'cuotas_texto',
            'pagina', 'posicion_pagina', 'url_publicacion', 'url_fuente', 'fecha_extraccion']
limpio = unicas[columnas].reset_index(drop=True)
calidad = pd.DataFrame({'variable': limpio.columns,
    'tipo_pandas': [str(limpio[c].dtype) for c in limpio],
    'nulos': [int(limpio[c].isna().sum()) for c in limpio],
    'porcentaje_nulos': [limpio[c].isna().mean() * 100 for c in limpio],
    'valores_distintos': [limpio[c].nunique() for c in limpio]})
print('Filas originales:', len(raw))
print('Filas para revisión:', len(revision))
print('Repeticiones retiradas:', int(repetidas.sum()))
print('Dimensiones finales:', limpio.shape)
print(calidad.to_string(index=False))

## Verificación y exportación
El dataset principal mantiene más de seis atributos, variables numéricas y categóricas, al menos mil ofertas identificadas y como máximo 10 % de nulos por columna. Las columnas de página y posición son metadatos; no se presentan como características comerciales del producto. Los precios extremos se conservan: no se eliminan solo por ser altos o bajos.

In [ ]:
assert len(limpio) >= 1000, 'No se alcanza el mínimo de registros.'
assert len(limpio.columns) >= 6
assert limpio.id_publicacion.is_unique
assert calidad.porcentaje_nulos.le(10).all(), 'Hay columnas que superan el 10 % de nulos.'
assert limpio.precio_actual.gt(0).all()
assert limpio.numero_cuotas.dropna().gt(0).all()
assert limpio.monto_cuota.dropna().gt(0).all()
assert len(raw) == len(revision) + int(repetidas.sum()) + len(limpio)
limpio.to_csv(SALIDA / 'mercadolibre_limpio.csv', index=False, encoding='utf-8', date_format='%Y-%m-%d %H:%M:%S')
calidad.to_csv(SALIDA / 'calidad_por_columna.csv', index=False, encoding='utf-8')
releido = pd.read_csv(SALIDA / 'mercadolibre_limpio.csv', encoding='utf-8')
assert releido.shape == limpio.shape
assert releido.id_publicacion.is_unique
assert releido.isna().mean().le(.10).all()
resumen = limpio.groupby('categoria').agg(registros=('id_publicacion', 'size'),
    precio_minimo=('precio_actual', 'min'), precio_mediano=('precio_actual', 'median'),
    precio_promedio=('precio_actual', 'mean'), precio_maximo=('precio_actual', 'max')).round(2)
resumen.to_csv(SALIDA / 'resumen_por_categoria.csv', encoding='utf-8')
print(resumen.to_string())
print('Validaciones aprobadas y CSV reabierto correctamente.')

## Texto de la sección Dataset preliminar
La siguiente celda genera un texto con los resultados reales de esta ejecución, una tabla de nulos y una muestra para incorporar al informe. El archivo `notas_integracion.md` explica qué deben ajustar los responsables de las otras partes.

In [ ]:
def tabla(frame):
    encabezado = '| ' + ' | '.join(map(str, frame.columns)) + ' |'
    separador = '| ' + ' | '.join(['---'] * len(frame.columns)) + ' |'
    filas = ['| ' + ' | '.join(str(x).replace('|', '/') for x in fila) + ' |' for fila in frame.itertuples(index=False, name=None)]
    return '\n'.join([encabezado, separador] + filas)

texto = f"""# 4. Dataset preliminar

## 4.1 Descripción de la muestra
Se procesaron {len(raw):,} observaciones crudas de Mercado Libre Perú, obtenidas entre {df.fecha_extraccion.min()} y {df.fecha_extraccion.max()}, según las marcas temporales del scraper. El archivo no registra zona horaria. Las búsquedas de origen fueron celulares, laptops, televisores, refrigeradoras y audifonos. El dataset final contiene {len(limpio):,} ofertas con identificadores únicos y {len(limpio.columns)} columnas.

La unidad de observación es la oferta identificada en la URL, conservando su última aparición en el archivo. La muestra procede de páginas de resultados y no es aleatoria ni representativa de todo el mercado peruano. La categoría es la búsqueda de origen y puede incluir accesorios o productos relacionados. Los precios describen ofertas, no transacciones realizadas.

## 4.2 Preparación y control de calidad
Se normalizaron espacios, se transformaron precios a formato numérico, se estandarizó la moneda a PEN y se convirtieron las fechas a datetime. Se extrajeron el número de cuotas y el monto por cuota de los textos observados, manteniendo los nulos cuando la información no estaba disponible.

El identificador extraído originalmente por el scraper se conserva como id_extraido. Para id_publicacion se tomó el parámetro wid de la URL o, cuando estaba ausente, el código de una ruta directa sin /p/. Se separaron {len(revision)} filas sin campos esenciales válidos y se retiraron {int(repetidas.sum())} observaciones repetidas, conservando la última por fecha y orden de archivo. Los registros separados se guardaron como evidencia para revisión.

No se encontraron duplicados de fila completa en el CSV original ({int(raw.duplicated().sum())}); las repeticiones se detectaron por identificador de oferta, aunque sus posiciones, fechas o enlaces de seguimiento fueran diferentes.

Se excluyeron precio_anterior_texto y vendedor_texto del dataset principal debido a sus ausencias originales de {raw.precio_anterior_texto.isna().mean()*100:.2f} % y {raw.vendedor_texto.isna().mean()*100:.2f} %, respectivamente. Ambas columnas permanecen en el CSV crudo. No se imputaron precios anteriores, vendedores ni descuentos. La falta de un precio anterior no permite concluir que el descuento sea cero.

## 4.3 Variables y cobertura
Las variables comerciales incluyen titulo, categoria, moneda, precio_actual, numero_cuotas, monto_cuota y cuotas_texto. Los identificadores, páginas, posiciones, URL y fecha proporcionan trazabilidad. El dataset incluye variables numéricas y categóricas, supera los 1000 registros y los seis atributos mínimos, y ninguna columna supera el 10 % de valores nulos. La tabla presenta los tipos en pandas antes de exportar; CSV no conserva tipos y deben declararse al volver a cargarlo.

{tabla(calidad.round(2))}

## 4.4 Estadísticos descriptivos
Todos los precios se expresan en soles (PEN). Se conservaron los valores extremos porque su magnitud por sí sola no prueba un error. Las diferencias entre categorías no equivalen a diferencias entre productos comparables.

{tabla(resumen.reset_index())}

## 4.5 Vista previa
Se muestra una selección de columnas de las primeras cinco filas. El CSV contiene la totalidad de los atributos descritos.

{tabla(limpio[['id_publicacion', 'categoria', 'titulo', 'precio_actual', 'numero_cuotas']].head())}

## 4.6 Limitaciones
No se extrajeron calificación, número de opiniones, condición, marca ni envío gratuito. Por tanto, estos datos no permiten analizar esas características ni la reputación del vendedor. Tampoco permiten calcular descuentos de manera completa. La deduplicación conserva una sola observación por oferta y no constituye una serie histórica. La identidad se recuperó de las URL guardadas y no se verificó mediante consultas adicionales al sitio.
"""
(SALIDA / 'Dataset_preliminar.md').write_text(texto, encoding='utf-8')
print('Texto del informe guardado en', SALIDA / 'Dataset_preliminar.md')

## Descargar los resultados
El ZIP contiene el dataset limpio, el texto para el informe y la evidencia de las decisiones de limpieza.

In [ ]:
from shutil import make_archive
from google.colab import files

(SALIDA / 'notas_integracion.md').write_text('# Integración con el trabajo del grupo\n\n## Archivos para entregar al grupo\n- ACD_PARTE_3.ipynb: limpieza documentada, con resultados ejecutados. Integrar después del scraper de la Parte 2.\n- resultados_parte3/mercadolibre_limpio.csv: dataset principal en UTF-8.\n- resultados_parte3/Dataset_preliminar.md: sección 4 para copiar al informe Word y luego exportar con el informe final.\n- resultados_parte3/calidad_por_columna.csv y resumen_por_categoria.csv: evidencia y apoyo para la Parte 4.\n- filas_para_revision.csv y duplicados_retirados.csv: trazabilidad; no mezclar de nuevo con el CSV limpio.\n\n## Ajustes necesarios en el informe original\n1. El objetivo específico 2 y las secciones de estrategia deben describir Selenium y BeautifulSoup, navegación con el botón Siguiente y hasta veinte páginas por categoría. El notebook entregado usa esas técnicas; no corresponde afirmar que extrae todo con requests ni que recorrió un mínimo de veinte páginas en cada categoría.\n2. El objetivo específico 4 promete calificación, opiniones, condición y envío gratuito, que no aparecen en el CSV crudo. Ajustar los objetivos a las variables reales o coordinar una nueva extracción si el grupo decide conservar esa promesa. La consigna exige seis atributos numéricos y categóricos en conjunto, no cuatro de cada tipo.\n3. La sección 6.4 y el anexo A.4 son propuestas: el diccionario final debe usar las 14 columnas del CSV limpio. No presentar precio anterior, vendedor o descuento como variables completas del archivo final.\n4. La Parte 4 debe documentar que id_extraido es el código original y puede identificar un producto de catálogo; id_publicacion se recupera de wid o de una ruta directa. Las URL originales se preservan para auditar esta decisión.\n5. La comprobación HTTP 200 de robots.txt del notebook no demuestra por sí sola que se revisaron las reglas aplicables. El responsable de adquisición debe conservar la evidencia de la revisión que declare en el informe.\n6. La Parte 5 debe completar identificación, nombres, estados e índice y adaptar el texto entregado al formato Word oficial. Esta entrega cubre la Parte 3, no el diccionario ni la consolidación final.\n', encoding='utf-8')
ruta_zip = make_archive('/content/entrega_parte3', 'zip', root_dir=SALIDA)
print('Limpieza terminada. Descargando resultados...')
files.download(ruta_zip)